In [1]:
label2id = {"left": 0, "center": 1, "right": 2}
id2label = {v: k for k, v in label2id.items()}

In [2]:
from transformers import RobertaTokenizer

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )


/Users/sarah/Documents/SMU/NLP/NLP_Project/venv_nlp_project/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [36]:
# load data from our file
from data_loader import load_split

data_dir = "/Users/sarah/Documents/SMU/NLP/Article-Bias-Prediction-main/data"

X_train, y_train = load_split(data_dir, "media", "train")
X_val, y_val = load_split(data_dir, "media", "valid")
X_test, y_test = load_split(data_dir, "media", "test")

print(len(X_train), len(X_val), len(X_test))

26590 2356 1300


In [44]:
import pandas as pd

train_df = pd.DataFrame({
    "text": X_train,
    "label": y_train
})

val_df = pd.DataFrame({
    "text": X_val,
    "label": y_val
})

test_df = pd.DataFrame({
    "text": X_test,
    "label": y_test
})


In [5]:
print(y_train[:5])

[2, 1, 0, 1, 0]


In [41]:
print(train_df["label"].value_counts())
print(val_df["label"].value_counts())
print(test_df["label"].value_counts())


label
2    10241
0     8861
1     7488
Name: count, dtype: int64
label
0    1640
1     618
2      98
Name: count, dtype: int64
label
2    599
0    402
1    299
Name: count, dtype: int64


In [48]:
train_df.head()

,text,label
0,President Trump and Senate Minority Leader Chu...,2
1,“ The 360 ” shows you diverse perspectives on ...,1
2,LOS ANGELES — Actress Rosario Dawson took the ...,0
3,President Donald Trump said on Friday that he ...,1
4,Washington ( CNN ) Donald Trump became the 45t...,0


In [63]:
# sample datasets down to get faster training & eval
def quick_balanced_sample(df, n_per_class=500, random_state=42):
    samples = []
    
    for label in df["label"].unique():
        subset = df[df["label"] == label]
        
        sampled_subset = subset.sample(
            n=min(len(subset), n_per_class),
            random_state=random_state
        )
        
        samples.append(sampled_subset)
    
    # Combine and shuffle
    balanced_df = (
        pd.concat(samples)
        .sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
    )
    
    return balanced_df

In [64]:
train_sampled = quick_balanced_sample(train_df, 500)
val_sampled   = quick_balanced_sample(val_df, 200)
test_sampled  = quick_balanced_sample(test_df, 200)


In [75]:
print(train_sampled["label"].value_counts())
print(val_sampled["label"].value_counts())
print(test_sampled["label"].value_counts())


label
0    500
2    500
1    500
Name: count, dtype: int64
label
1    200
0    200
2     98
Name: count, dtype: int64
label
2    200
0    200
1    200
Name: count, dtype: int64


In [65]:
train_sampled.head()

,text,label
0,"Julian Assange insists , against all evidence ...",0
1,This is a rush transcript . Copy may not be in...,0
2,Iran on Saturday reportedly released Washingto...,2
3,Not even six months into its existence ObamaCa...,2
4,Is the Obamacare coalition finally cracking ? ...,2


In [66]:
from datasets import Dataset

# train_dataset = Dataset.from_pandas(train_df)
# val_dataset = Dataset.from_pandas(val_df)
# test_dataset = Dataset.from_pandas(test_df)

train_dataset = Dataset.from_pandas(train_sampled)
val_dataset = Dataset.from_pandas(val_sampled)
test_dataset = Dataset.from_pandas(test_sampled)



In [67]:
from datasets import DatasetDict

dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

In [68]:
from transformers import RobertaTokenizer

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

dataset = dataset.map(tokenize, batched=True)

Map: 100%|██████████| 600/600 [00:00<00:00, 2199.67 examples/s]


In [69]:
dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

In [70]:
from transformers import RobertaForSequenceClassification

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 8889.77it/s]
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [71]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=1)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted"  # or "macro"
    )
    
    acc = accuracy_score(labels, predictions)
    
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [79]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    #evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    #tokenizer=tokenizer
    compute_metrics=compute_metrics
)

#trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [80]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    #tokenizer=tokenizer
)

In [ ]:
trainer.train()

Step,Training Loss
500,0.290014


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.17it/s]
/Users/sarah/Documents/SMU/NLP/NLP_Project/venv_nlp_project/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.74it/s]
/Users/sarah/Documents/SMU/NLP/NLP_Project/venv_nlp_project/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.08it/s]
/Users/sarah/Documents/SMU/NLP/NLP_Project/venv_nlp_project/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(

RuntimeError: on_train_begin must be called before on_evaluate

In [77]:
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)

def classify(text):
    model.eval()
    
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )
    
    # 🔑 Move inputs to same device as model
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    logits = outputs.logits
    predicted_class_id = torch.argmax(logits, dim=1).item()
    
    return id2label[predicted_class_id]

In [82]:
print(classify("The policy is bad because it's extrememly bad for the environent. This is a regressive policy that will lead to more pollution and harm our planet."))
# → "right" / "left" / "center"

right


In [83]:
predictions = trainer.predict(dataset["test"])

import numpy as np
from sklearn.metrics import classification_report

y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)

print(classification_report(y_true, y_pred, target_names=["left", "center", "right"]))


              precision    recall  f1-score   support

        left       0.54      0.67      0.60       200
      center       0.53      0.43      0.48       200
       right       0.60      0.57      0.58       200

    accuracy                           0.56       600
   macro avg       0.56      0.56      0.55       600
weighted avg       0.56      0.56      0.55       600



In [84]:
dataset["test"].column_names


['text', 'label', 'input_ids', 'attention_mask']